# Doctor-Assistant — Full System Test

This notebook exercises **every routing path** the system knows about using a mixed panel of test scans:

| # | Scenario | Modality | Body part | Expert(s) expected |
|---|---|---|---|---|
| 1 | Normal chest X-ray | X-ray | Chest | chest_xray + chest_xrv |
| 2 | Abnormal CXR — real NIH X-ray | X-ray | Chest | chest_xray + chest_xrv |
| 3 | BiomedCLIP backbone (same scan) | X-ray | Chest | chest_xray (BiomedCLIP) |
| 4 | MAIRA-2 grounded reporter | X-ray | Chest | chest_maira2 |
| 5 | CT abdomen — organ volumetry | CT | Abdomen | ct_totalsegmentator |
| 6 | Router fallback — unknown body part | X-ray | UNKNOWN | chest_xray + chest_xrv (fallback) |
| 7 | Router miss — no expert registered | MRI | Brain | RoutingError |

**The chest reader.** Two classifiers register under (XRAY, CHEST) and their findings pool:
- `chest_xray` — *our* DenseNet (trained on a Colab budget; also drives Grad-CAM). Useful for the architecture, but weakly calibrated.
- `chest_xrv` — **pretrained TorchXRayVision** DenseNet, trained on NIH + CheXpert + MIMIC + PadChest. Calibrated multi-label scores that actually cross threshold on real findings. This is the reader that makes scenarios 1/2/6 light up. Local weights, downloaded once, no API. Toggle with `RUN_XRV` (cell 5).

**Data sources (no authentication needed):**
- Chest X-rays: **real NIH ChestX-ray14** samples — full-resolution 1024px images with ground-truth labels, streamed from the HuggingFace Hub by `scripts/fetch_samples.py` (public, no credentials; a curated dozen downloads in seconds)
- CT abdomen: synthetic NIfTI with realistic Hounsfield values (no download needed)
- Scenarios 3 / 4 / 5 are opt-in (large model downloads) — set the flags in cell 5

> **Why real samples + a pretrained reader:** real findings only fire when *both* are right. The images must be the native ~1024px NIH domain (ChestMNIST's ≤224px mush suppresses findings), **and** the classifier must be well-calibrated. Our own checkpoint ranks okay (AUC ~0.74) but squashes every probability below ~0.15, so nothing crosses 0.4. TorchXRayVision's pretrained weights give calibrated scores on the same images — so the report finally reflects the pathology.

**Runtime:** L4 GPU recommended. CPU works for everything except MAIRA-2.


## 0 · GPU check

In [1]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU : {gpu.name}  ({gpu.total_memory/1e9:.1f} GB VRAM)')
else:
    print('No GPU — running on CPU (MAIRA-2 will be very slow)')
print(f'PyTorch: {torch.__version__}')

GPU : NVIDIA L4  (23.7 GB VRAM)
PyTorch: 2.11.0+cu128


## 1 · Pull codebase

In [2]:
import os, sys, importlib
!git -C /content/doctor_assistant pull --ff-only
REPO_URL = "https://github.com/Hamza09Hamza/doctor_assistant.git"
REPO_DIR = "/content/doctor_assistant"

if os.path.isdir(REPO_DIR):
    ret = os.system(f"git -C {REPO_DIR} pull --ff-only -q")
    if ret != 0:
        raise RuntimeError("git pull failed — check network.")
else:
    ret = os.system(f"git clone -q {REPO_URL} {REPO_DIR}")
    if ret != 0 or not os.path.isdir(REPO_DIR):
        raise RuntimeError("git clone failed — check network.")

sys.path.insert(0, REPO_DIR)

for mod in ["experts", "routing", "reporting", "pipeline", "core"]:
    if importlib.util.find_spec(mod) is None:
        raise ImportError(f"Module '{mod}' not found — clone may be incomplete.")

print(f"Repo ready.  Contents: {os.listdir(REPO_DIR)}")

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 7.35 KiB | 1.84 MiB/s, done.
From https://github.com/Hamza09Hamza/doctor_assistant
   5c1c31b..3c56ec3  main       -> origin/main
Updating 5c1c31b..3c56ec3
Fast-forward
 notebooks/system_test.ipynb | 556 +++++++++++++++++---------------------------
 1 file changed, 217 insertions(+), 339 deletions(-)
Repo ready.  Contents: ['.gitignore', 'requirements.txt', 'configs', 'evaluation', 'pipeline.py', 'data', 'models', 'scripts', 'notebooks', 'reporting', 'explainability', 'training', 'samples', 'setup.md', 'routing', 'experts', 'preprocessing', 'ingest', 'core', '.git']


## 2 · Install dependencies

In [3]:
%%capture
# Core stack (always needed)
!pip install -q \
    'monai>=1.3' 'nibabel>=5.2' 'timm>=0.9' \
    'scikit-learn>=1.4' 'scikit-image>=0.22' \
    'pydantic>=2.6' 'Pillow>=10'

# Real, in-domain test data — full-resolution NIH ChestX-ray14 samples streamed from
# the HuggingFace Hub by scripts/fetch_samples.py. Light + safe. Public, no credentials.
!pip install -q 'datasets>=2.18' 'huggingface_hub>=0.23'

# Strong PRETRAINED chest classifier — TorchXRayVision. A light, pure-Python wrapper over
# pretrained DenseNet weights; it does NOT disturb Colab's numpy/scipy. This is the reader
# that makes scenarios 1/2/6 show real findings. Local weights, downloaded once, no API.
!pip install -q torchxrayvision

# MAIRA-2 grounded reporter (Scenario 4) + local Phi-3 LLM report drafting (RUN_LLM).
# Both run fully local (weights download once, no API key, no inference network call).
!pip install -q 'transformers>=4.46' protobuf sentencepiece accelerate bitsandbytes

# ── BiomedCLIP (Scenario 3) — only if RUN_BIOMEDCLIP=True ─────────────────────
# !pip install -q open_clip_torch

# ── TotalSegmentator (Scenario 5 / CT) — DO NOT install in this runtime ───────
# It pins its own numpy / scipy / torch (via nnU-Net) and REWRITES Colab's scientific
# stack, which breaks `datasets`, scipy, and the whole chest pipeline in the SAME session
# (the "cannot import name '_center' from 'numpy._core.umath'" error). To run CT, open a
# SEPARATE Colab runtime: uncomment the next line AND set RUN_CT=True (cell 5) there.
# !pip install -q TotalSegmentator

print('Dependencies installed.')


## 3 · Mount Drive (to load the trained checkpoint)

In [4]:
from google.colab import drive
drive.mount('/content/drive')

CKPT_PATH = '/content/drive/MyDrive/doctor_assistant/checkpoints/chest_xray/best.pt'
has_checkpoint = os.path.isfile(CKPT_PATH)
print(f'Trained checkpoint on Drive: {has_checkpoint}')
if not has_checkpoint:
    print('  → Will use random weights (wiring test still valid; AUC numbers are random)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Trained checkpoint on Drive: True


## 4 · Test-scenario flags

Flip the ones you want to run. Scenarios 3-5 each download a large model on first run.

In [ ]:
# ── Which chest reader(s) to use ─────────────────────────────────────────────
# RUN_XRV is the important one: a STRONG pretrained chest classifier
# (TorchXRayVision DenseNet, trained on NIH+CheXpert+MIMIC+PadChest) with calibrated
# scores that actually cross threshold on real findings. It runs ALONGSIDE our own
# trained DenseNet (their findings pool), so scenarios 1/2/6 finally show real results
# instead of everything-below-0.4. Local weights, no API.
RUN_XRV          = True    # pretrained TorchXRayVision classifier (recommended ON)

# ── Optional extra experts ───────────────────────────────────────────────────
# NOTE on BiomedCLIP: it does NOT add a scenario alongside the trained model — it
# REPLACES the DenseNet backbone for the whole run, and its classification head is
# untrained, so scores become wiring-only. Leave False unless you want that demo.
RUN_BIOMEDCLIP   = False   # Scenario 3: swap backbone to BiomedCLIP ViT-B/16

# MAIRA-2 is a GATED HuggingFace model: you must (1) request access at
# huggingface.co/microsoft/maira-2 and be approved, then (2) provide an HF token
# (Colab: add HF_TOKEN as a secret, or run `huggingface_hub.login()`). Without that it
# 401s. Left OFF so it can't block the chest pipeline; the pipeline now also skips any
# reader that fails instead of crashing the scan.
RUN_MAIRA2       = False   # Scenario 4: MAIRA-2 grounded reporter (needs HF access + token)

# CT / TotalSegmentator MUST run in its own Colab runtime — it rewrites numpy/scipy and
# breaks the chest pipeline if installed alongside it (see cell 3). Keep False here; flip
# to True only in a dedicated CT runtime where you also uncommented its install in cell 3.
RUN_CT           = False    # Scenario 5: TotalSegmentator for CT abdomen (separate runtime)

# ── Report drafting ──────────────────────────────────────────────────────────
# False → deterministic template reports (fast, no download).
# True  → local LLM (Phi-3 Mini, 4-bit ~2GB) polishes the prose. Fully local,
#         no API key. Requires the LLM deps in cell 3 (now uncommented).
RUN_LLM          = True

# ── CXR model config (must match what was trained) ───────────────────────────
BACKBONE   = 'timm:densenet121'
IMAGE_SIZE = 320

# Overrides if BiomedCLIP is requested (it has a fixed 224px input)
if RUN_BIOMEDCLIP:
    BACKBONE   = 'biomedclip:'
    IMAGE_SIZE = 224

print(f'Backbone: {BACKBONE}  |  image_size: {IMAGE_SIZE}')
print(f'XRV(pretrained): {RUN_XRV}  |  MAIRA-2: {RUN_MAIRA2}  |  '
      f'TotalSegmentator: {RUN_CT}  |  LLM: {RUN_LLM}')

## 5 · Download mixed test data

**Real NIH ChestX-ray14 samples.** `scripts/fetch_samples.py` streams a curated dozen of
the *original* full-resolution images (1024px PNGs) with their ground-truth labels from
the HuggingFace Hub (`BahaaEldin0/NIH-Chest-Xray-14`) — public, no credentials, no 30 GB
download. These are the exact domain the model was trained on, so its scores are
meaningful — unlike ChestMNIST, whose ≤224px downsampling suppresses real findings.
We also generate a synthetic CT NIfTI (realistic Hounsfield values) for the CT routing
scenario.


In [6]:
# ── Real, in-domain test data: full-resolution NIH ChestX-ray14 ─────────────────
# The chest model was trained on native ~1024px NIH X-rays. ChestMNIST is the SAME
# images pre-crushed to 28-224px; upscaling one to 320px cannot recover the fine
# texture the model keys on, so every score collapses below threshold and the demo
# looks broken. Here we pull the ORIGINAL images (with ground-truth labels) so the
# model sees the exact domain it was trained on — and findings actually fire.
#
# Source: BahaaEldin0/NIH-Chest-Xray-14 on HuggingFace (1024px PNGs, the canonical
# 14-label vocabulary). Public, no credentials, streams a curated dozen in seconds.
import csv, os, importlib.util
import torch
from ingest.loaders import load_scan
from preprocessing.transforms import PreprocessConfig, build_preprocess
from core.enums import BodyPart, Modality

SAMPLES_DIR = os.path.join(REPO_DIR, 'samples', 'chest_xray')

# `datasets` should already be installed by cell 5; install it here too in case this
# cell is run on a fresh runtime where cell 5 was skipped.
try:
    import datasets  # noqa: F401
except ImportError:
    print('Installing `datasets` ...')
    !pip install -q 'datasets>=2.18' 'huggingface_hub>=0.23'

# Run the fetch IN-KERNEL (not via a subprocess) so it uses this kernel's installed
# `datasets` and any error is visible right here. Load the script by file path so we
# always pick up the version just pulled from GitHub.
_fs_path = os.path.join(REPO_DIR, 'scripts', 'fetch_samples.py')
_spec = importlib.util.spec_from_file_location('fetch_samples', _fs_path)
_fs = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_fs)

rc = _fs.fetch_samples(SAMPLES_DIR)   # idempotent: skips if already present
if rc != 0:
    raise RuntimeError(f'fetch_samples failed (rc={rc}) — see the message above.')

# Exact inference preprocessing the expert/training used: scale -> resize(IMAGE_SIZE)
# -> 3 channels. ScaleIntensity is min-max so re-applying it in expert.predict() is a
# no-op; the tensor we hand out is already a finished (3, IMAGE_SIZE, IMAGE_SIZE) input.
_pre = build_preprocess(
    PreprocessConfig(spatial_size=(IMAGE_SIZE, IMAGE_SIZE), in_channels=3, intensity='scale'),
    train=False,
)


class RealCXRSamples:
    """Tiny dataset over the fetched NIH PNGs.

    __getitem__(i) -> (preprocessed 3xHxW tensor, set-of-true-label-names).
    Preprocessing matches training exactly, so the model sees identical inputs to
    eval time. The label set is the ground truth from the dataset (empty == normal).
    """

    def __init__(self, samples_dir):
        manifest = os.path.join(samples_dir, 'manifest.csv')
        self.items = []
        with open(manifest, newline='') as f:
            for row in csv.DictReader(f):
                labels = set(filter(None, row['labels'].split('|')))
                path = os.path.join(samples_dir, 'images', row['filename'])
                self.items.append((path, labels))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, labels = self.items[i]
        scan = load_scan(path, modality=Modality.XRAY, body_part=BodyPart.CHEST)
        x = _pre(scan.data)
        if hasattr(x, 'as_tensor'):   # MONAI MetaTensor -> plain tensor
            x = x.as_tensor()
        return x.contiguous().float(), labels

    def true_labels(self, i):
        return self.items[i][1]


chest_ds = RealCXRSamples(SAMPLES_DIR)
print(f'\nReal NIH samples: {len(chest_ds)} images from {SAMPLES_DIR}\n')
for i in range(len(chest_ds)):
    name = os.path.basename(chest_ds.items[i][0])
    true = sorted(chest_ds.true_labels(i)) or ['No Finding']
    print(f'  [{i:2}] {name:26} true = {true}')

# Scenario 1 uses a normal (no findings); scenario 2 uses the first abnormal sample.
normal_idx   = next((i for i in range(len(chest_ds)) if not chest_ds.true_labels(i)), 0)
abnormal_idx = next((i for i in range(len(chest_ds)) if chest_ds.true_labels(i)), 0)
print(f'\nnormal_idx={normal_idx}  abnormal_idx={abnormal_idx}')


Streaming BahaaEldin0/NIH-Chest-Xray-14 [test] from HuggingFace (no credentials needed)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Resolving data files:   0%|          | 0/73 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/73 [00:00<?, ?it/s]

  saved 00020588_000.png             true=['Cardiomegaly']
  saved 00009992_002.png             true=['Effusion']
  saved 00010035_000.png             true=['Atelectasis']
  saved 00020827_013.png             true=['Infiltration']
  saved 00015310_004.png             true=['Mass']
  saved 00013737_001.png             true=['Nodule']
  saved 00000118_005.png             true=['Pneumothorax']
  saved 00014022_073.png             true=['Consolidation']
  saved 00008139_000.png             true=['No Finding']
  saved 00005768_006.png             true=['No Finding']

Done: 10 images (8 abnormal, 2 normal) -> /content/doctor_assistant/samples/chest_xray
Manifest: /content/doctor_assistant/samples/chest_xray/manifest.csv


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.



Real NIH samples: 10 images from /content/doctor_assistant/samples/chest_xray

  [ 0] 00020588_000.png           true = ['Cardiomegaly']
  [ 1] 00009992_002.png           true = ['Effusion']
  [ 2] 00010035_000.png           true = ['Atelectasis']
  [ 3] 00020827_013.png           true = ['Infiltration']
  [ 4] 00015310_004.png           true = ['Mass']
  [ 5] 00013737_001.png           true = ['Nodule']
  [ 6] 00000118_005.png           true = ['Pneumothorax']
  [ 7] 00014022_073.png           true = ['Consolidation']
  [ 8] 00008139_000.png           true = ['No Finding']
  [ 9] 00005768_006.png           true = ['No Finding']

normal_idx=8  abnormal_idx=0


In [7]:
import numpy as np
import nibabel as nib

# ── Synthetic CT abdomen NIfTI ─────────────────────────────────────────────
# 256×256×100 volume with Hounsfield values typical of an abdominal CT.
# Liver ~60 HU, spleen ~40 HU, kidney ~30 HU, fat ~-100 HU, air ~-1000 HU.
# TotalSegmentator needs a NIfTI on disk with real voxel spacing.

SYNTH_CT_PATH = '/content/synth_abdomen_ct.nii.gz'

np.random.seed(42)
vol = np.full((256, 256, 100), -100.0, dtype=np.float32)   # fat background

# Air outside body
cx, cy = 128, 128
for z in range(100):
    Y, X = np.ogrid[:256, :256]
    body = (X - cx)**2 + (Y - cy)**2 < 110**2
    vol[:, :, z][~body] = -1000.0

# Liver (right lobe, HU ~40-80)
vol[60:180, 80:200, 30:80] += np.random.uniform(40, 80, (120, 120, 50)).astype(np.float32) + 100
# Spleen (left side, HU ~35-55)
vol[60:130, 50:110, 35:75] += np.random.uniform(35, 55, (70, 60, 40)).astype(np.float32) + 100
# Aorta (central, HU ~200+)
vol[115:145, 115:145, 10:90] += 300.0

affine = np.diag([1.5, 1.5, 2.5, 1.0])  # 1.5mm×1.5mm×2.5mm voxels
img = nib.Nifti1Image(vol, affine)
nib.save(img, SYNTH_CT_PATH)
print(f'Synthetic CT saved: {SYNTH_CT_PATH}  shape={vol.shape}  spacing=(1.5,1.5,2.5)mm')


Synthetic CT saved: /content/synth_abdomen_ct.nii.gz  shape=(256, 256, 100)  spacing=(1.5,1.5,2.5)mm


## 6 · Build the expert panel + pipeline

In [8]:
import torch
from experts import build_chest_xray_expert, build_default_registry
from routing import ModalityRouter
from reporting import GridZoneLocalizer, Verifier, GuidelineEngine, Reporter
from pipeline import Pipeline
from core.enums import BodyPart, Modality
from data.chest_xray14 import CHESTXRAY14_LABELS
from models.experts import strip_orig_mod

# ── 1. Chest expert (trained weights from Drive, or random for wiring test) ──
# This is OUR trained DenseNet. It also drives Grad-CAM later (cell 15). With the weak
# checkpoint its own scores stay below threshold — the pretrained TorchXRayVision reader
# (added below when RUN_XRV=True) is what actually surfaces findings.
chest_expert = build_chest_xray_expert(
    backbone=BACKBONE, image_size=IMAGE_SIZE, pretrained=not RUN_BIOMEDCLIP
).to(device).eval()

if has_checkpoint and not RUN_BIOMEDCLIP:
    ckpt = torch.load(CKPT_PATH, map_location=device)
    chest_expert.load_state_dict(strip_orig_mod(ckpt['model']))
    # The head's logit order is load-bearing: index i means class_names[i]. The
    # checkpoint stores the exact class_names it was trained with, so adopt them —
    # this keeps findings correctly labeled even if a source label tuple ever drifts.
    if ckpt.get('class_names'):
        chest_expert.class_names = list(ckpt['class_names'])
    print(f"Chest expert: loaded from checkpoint  "
          f"(epoch {ckpt['epoch']}, AUC {ckpt.get('best_metric', float('nan')):.4f})")
    print(f"  class order: {chest_expert.class_names}")
elif RUN_BIOMEDCLIP:
    # The DenseNet checkpoint is architecturally incompatible with the BiomedCLIP
    # backbone, and BiomedCLIP's classification head is untrained — so any scores it
    # produces are a wiring demo, NOT diagnostic. Scenarios 1/2/6 also run on this
    # untrained backbone while RUN_BIOMEDCLIP=True.
    print('Chest expert: BiomedCLIP backbone, UNTRAINED head — scores are wiring-only, '
          'not diagnostic (DenseNet checkpoint not loaded).')
else:
    print('Chest expert: random weights (no checkpoint found on Drive)')

# ── 2. Assemble the expert registry ──────────────────────────────────────────
# include_xrv adds the pretrained TorchXRayVision classifier under (XRAY, CHEST). The
# registry keeps a list per niche, so it runs ALONGSIDE chest_expert and the orchestrator
# pools both readers' findings — the strong pretrained scores are what cross threshold.
registry = build_default_registry(
    chest_expert=chest_expert,
    include_xrv=RUN_XRV,
    include_maira2=RUN_MAIRA2,
    include_ct=RUN_CT,
)
chest_readers = [e.name for e in registry.match(Modality.XRAY, BodyPart.CHEST)]
niches = sorted((m.value, b.value) for (m, b) in registry._by_niche)
print(f'Chest (XRAY, CHEST) readers: {chest_readers}')
print(f'Registered niches: {niches}')

# ── 3. Reporter — local LLM (Phi-3) only when RUN_LLM=True, else template ──────
reporter = Reporter(auto_llm=RUN_LLM)
print(f'Reporter LLM: {"Phi-3 Mini (local, 4-bit)" if reporter.llm else "template only"}')

# ── 4. Build the orchestrator ─────────────────────────────────────────────────
# strict=False so we can test the fallback path (scenario 6)
router = ModalityRouter(registry, strict=False)
pipe   = Pipeline(
    router,
    reporter=reporter,
    verifier=Verifier(known_labels=list(CHESTXRAY14_LABELS)),
    guidelines=GuidelineEngine(),
    thresholds=0.4,          # lower threshold to surface more findings in the demo
    localizer=GridZoneLocalizer(),
)
print('Pipeline ready.')

model.safetensors:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

Chest expert: loaded from checkpoint  (epoch 18, AUC 0.7387)
  class order: ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']
Chest (XRAY, CHEST) readers: ['chest_xray', 'chest_xrv', 'chest_maira2']
Registered niches: [('xray', 'chest')]
Reporter LLM: Phi-3 Mini (local, 4-bit)
Pipeline ready.


## 7 · Helper: run one scan and print the full result

In [9]:
from core.types import Scan, ScanMetadata
from routing.router import RoutingError
import traceback

RESULTS = []   # collect for the summary table at the end

def run_scenario(label, data_tensor, modality, body_part, source_path=None, expect_error=False):
    print(f'\n{"="*72}')
    print(f'SCENARIO: {label}')
    print(f'  Modality={modality.value}  BodyPart={body_part.value}')
    print(f'{"="*72}')

    scan = Scan(
        data=data_tensor.to(device),
        meta=ScanMetadata(
            modality=modality,
            body_part=body_part,
            source_path=source_path,
        )
    )

    try:
        result = pipe.analyze_scan(scan)
    except RoutingError as e:
        print(f'  [RoutingError — expected={expect_error}]  {e}')
        RESULTS.append({
            'scenario': label,
            'modality': modality.value,
            'body_part': body_part.value,
            'experts': '— (routing error)',
            'findings': '—',
            'urgency': '—',
            'verifier': '—',
            'generator': '—',
        })
        return

    present = [f for f in result.findings if f.present]
    print(f'  Routed to  : {result.experts}')
    print(f'  Findings   : {[f.label for f in present] or "[none above threshold]"}')

    # Raw top scores (diagnostic) — what the model actually predicted, BEFORE the
    # threshold filter drops sub-threshold labels. Lets you tell "model saw nothing"
    # (scores ~0.05) apart from "just under the line" (scores ~0.38).
    raw = {}
    for p in result.predictions:
        raw.update(getattr(p, 'class_probs', {}) or {})
    if raw:
        top5 = sorted(raw.items(), key=lambda kv: kv[1], reverse=True)[:5]
        print('  Top scores : ' + ', '.join(f'{k}={v:.2f}' for k, v in top5))

    print(f'  Triage     : {result.triage_urgency.name}')
    print(f'  Report gen : {result.report.generator}')
    verif = result.verification
    verif_str = f'{"PASS" if verif.ok else "FAIL"} ({verif.grounded_fraction*100:.0f}% grounded)'
    print(f'  Verifier   : {verif_str}')
    if verif.flags:
        for flag in verif.flags:
            print(f'    ✗ {flag}')
    print()
    print(result.report.to_text())
    if result.recommendations:
        print('\nRECOMMENDATIONS:')
        for r in result.recommendations:
            print(f'  [{r.urgency.name}] {r.label}: {r.text}')

    RESULTS.append({
        'scenario': label,
        'modality': modality.value,
        'body_part': body_part.value,
        'experts': ', '.join(result.experts),
        'findings': ', '.join(f.label for f in present) or 'none',
        'urgency': result.triage_urgency.name,
        'verifier': verif_str,
        'generator': result.report.generator,
    })

print('Helper ready.')

Helper ready.


## 8 · Scenario 1 — Normal chest X-ray
A study where no pathology reaches the threshold. The pipeline should report no acute finding and ROUTINE urgency.

In [10]:
img_normal, label_normal = chest_ds[normal_idx]
run_scenario(
    label='1 · Normal chest X-ray',
    data_tensor=img_normal,
    modality=Modality.XRAY,
    body_part=BodyPart.CHEST,
)


SCENARIO: 1 · Normal chest X-ray
  Modality=xray  BodyPart=chest
If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/microsoft/maira-2.
401 Client Error. (Request ID: Root=1-6a2efaf6-256cf49132d2a93858805c21;553b9da2-6533-47e5-8a18-fac6670189ab)

Cannot access gated repo for url https://huggingface.co/microsoft/maira-2/resolve/main/config.json.
Access to model microsoft/maira-2 is restricted. You must have access to it and be authenticated to access it. Please log in.

## 9 · Scenario 2 — Abnormal chest X-ray (multiple findings)
A study with 2+ co-present pathologies. We expect a PROMPT or URGENT triage depending on which labels fire.

In [ ]:
img_abnormal, label_abnormal = chest_ds[abnormal_idx]
print(f'Ground-truth labels for this sample: {sorted(label_abnormal) or ["No Finding"]}')

run_scenario(
    label='2 · Abnormal CXR — real NIH X-ray',
    data_tensor=img_abnormal,
    modality=Modality.XRAY,
    body_part=BodyPart.CHEST,
)


## 10 · Scenario 3 — BiomedCLIP backbone  *(opt-in, RUN_BIOMEDCLIP=True)*

Shows that swapping the backbone is a single toggle — the rest of the pipeline is identical.
Skipped when `RUN_BIOMEDCLIP=False`.

In [ ]:
if RUN_BIOMEDCLIP:
    # BiomedCLIP expert is already the active `chest_expert` when backbone='biomedclip:'
    run_scenario(
        label='3 · BiomedCLIP backbone (same scan)',
        data_tensor=img_abnormal,
        modality=Modality.XRAY,
        body_part=BodyPart.CHEST,
    )
else:
    print('Scenario 3 skipped (RUN_BIOMEDCLIP=False). '
          'Set RUN_BIOMEDCLIP=True in cell 5 and re-run from there.')

## 11 · Scenario 4 — MAIRA-2 grounded reporter  *(opt-in, RUN_MAIRA2=True)*

MAIRA-2 runs **alongside** the trained classifier — both are registered under `(XRAY, CHEST)`,
so the router returns both and the orchestrator pools their findings. The grounded sentences
from MAIRA-2 are parsed into canonical `Finding`s with location boxes.

In [ ]:
if RUN_MAIRA2:
    run_scenario(
        label='4 · MAIRA-2 grounded reporter',
        data_tensor=img_abnormal,
        modality=Modality.XRAY,
        body_part=BodyPart.CHEST,
    )
else:
    print('Scenario 4 skipped (RUN_MAIRA2=False). MAIRA-2 is a GATED model: request '
          'access at huggingface.co/microsoft/maira-2, add an HF_TOKEN secret in Colab, '
          'then set RUN_MAIRA2=True (cell 5) and re-run from cell 7.')

## 12 · Scenario 5 — CT abdomen, organ volumetry  *(opt-in, RUN_CT=True)*

TotalSegmentator segments the synthetic NIfTI, returns liver/spleen/kidney volumes in mL.
The measurements are real physical units (voxel count × voxel spacing / 1000).

In [ ]:
if RUN_CT:
    import numpy as np
    import nibabel as nib
    ct_img = nib.load(SYNTH_CT_PATH)
    ct_tensor = torch.as_tensor(
        np.asarray(ct_img.dataobj).astype(np.float32)
    ).unsqueeze(0)   # add channel dim (1, D, H, W)

    run_scenario(
        label='5 · CT abdomen — TotalSegmentator organ volumetry',
        data_tensor=ct_tensor,
        modality=Modality.CT,
        body_part=BodyPart.ABDOMEN,
        source_path=SYNTH_CT_PATH,   # TotalSegmentator reads the file directly
    )
else:
    print('Scenario 5 skipped (RUN_CT=False). '
          'Uncomment TotalSegmentator in cell 3, set RUN_CT=True, re-run from cell 3.')


## 13 · Scenario 6 — Router fallback (UNKNOWN body part)

When the body part isn't known, the router tries to find *any* expert for the modality.
Here `(XRAY, UNKNOWN)` falls back to the chest expert because it handles X-ray.

In [ ]:
run_scenario(
    label='6 · Router fallback — XRAY with UNKNOWN body part',
    data_tensor=img_abnormal,
    modality=Modality.XRAY,
    body_part=BodyPart.UNKNOWN,
)

## 14 · Scenario 7 — Router miss (no expert registered for MRI Brain)

No brain MRI expert has been registered yet. The router (strict=False) exhausts all
fallbacks and raises `RoutingError`, which `run_scenario` catches and records.

In [ ]:
# Synthetic MRI volume (3D, single channel)
mri_tensor = torch.rand(1, 128, 128, 64)  # (C, H, W, D)

run_scenario(
    label='7 · Router miss — MRI Brain (no expert registered)',
    data_tensor=mri_tensor,
    modality=Modality.MRI,
    body_part=BodyPart.BRAIN,
    expect_error=True,
)

## 15 · Visualise — Grad-CAM on the two CXR scenarios

In [ ]:
import matplotlib.pyplot as plt
from explainability import GradCAM
from reporting import findings_from_classification
from core.types import Scan, ScanMetadata

def show_gradcam(title, img_tensor):
    scan = Scan(
        data=img_tensor.to(device),
        meta=ScanMetadata(modality=Modality.XRAY, body_part=BodyPart.CHEST),
    )
    pred  = chest_expert.predict(scan)
    cam   = GradCAM(chest_expert)
    hmaps = cam.for_labels(scan.data)
    findings = findings_from_classification(
        pred.class_probs, thresholds=0.4, localizer=GridZoneLocalizer()
    )
    present = [f for f in findings if f.present][:3]

    img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
    gray   = img_np[:, :, 0]

    n = len(present) + 1
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]
    axes[0].imshow(gray, cmap='gray'); axes[0].set_title(f'{title}\n(input)'); axes[0].axis('off')
    for ax, f in zip(axes[1:], present):
        hm = hmaps.get(f.label)
        ax.imshow(gray, cmap='gray')
        if hm is not None:
            ax.imshow(hm.numpy(), alpha=0.5, cmap='jet', vmin=0, vmax=1)
        ax.set_title(f'{f.label}\np={f.probability:.2f}  {f.location or ""}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    print(f'Top findings: {[(f.label, f.laterality, f.location) for f in present]}')

from reporting import GridZoneLocalizer
show_gradcam('Normal scan', img_normal)
show_gradcam('Abnormal scan', img_abnormal)

## 16 · Summary table

In [ ]:
import pandas as pd

df = pd.DataFrame(RESULTS)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 160)
print(df.to_string(index=False))

## 17 · Smoke test — run the repo's wiring check

Runs the same parser-core + routing checks that CI would run, but now with the real repo pulled from GitHub.

In [ ]:
!python {REPO_DIR}/scripts/smoke_system.py